In [0]:
from pyspark.sql import functions as F
from pyspark import pipelines as dp
from pyspark.sql import Window

In [0]:

CATALOG = spark.conf.get("catalog")
SILVER_SCHEMA = spark.conf.get("silver_schema")
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_netflix"


GOLD_SCHEMA = spark.conf.get("gold_schema")

DIM_TYPE = f"{CATALOG}.{GOLD_SCHEMA}.dim_type"
DIM_RATING = f"{CATALOG}.{GOLD_SCHEMA}.dim_rating"
DIM_RELEASE_YEAR = f"{CATALOG}.{GOLD_SCHEMA}.dim_release_year"

In [0]:
@dp.materialized_view(name=f"{GOLD_SCHEMA}.fact_tiles")
def dim_release_year():
    silver_df = spark.read.table(SILVER_TABLE)
    dim_type_df = spark.read.table(DIM_TYPE)
    dim_rating_df = spark.read.table(DIM_RATING)
    dim_release_year = spark.read.table(DIM_RELEASE_YEAR)

    return(
        silver_df
        .join(
            dim_type_df,
            on= "type",
            how = "left"
        )
        .join(
            dim_rating_df,
            on = ["rating", "audience_category"],
            how = "left"
        )
        .join(
            dim_release_year,
            on = ["release_year", "release_period"],
            how = "left"
        )
        .select(
            "show_id", 
            "type_key", 
            "rating_key",
            "release_year_key",
            F.lit(1).alias("title_count")
        )
    )